# Q3 - Bias and variance as measured quantities
Needs the saved split plus the Q1/Q2 frozen choices (`logs/frozen_tree_params.json`, `logs/final_B.json`, `logs/q2_frozen_boosting.json`). Missing choices are recomputed with the Q1/Q2 rules. **No tuning is done here.** The 30 refits run in parallel (`N_JOBS`); expect minutes on a multi-core PC and up to about an hour on one core.

In [ ]:
# ---- CONFIG ----
ROLL_NO = "0202"
SEED = int("".join(ch for ch in ROLL_NO if ch.isdigit()))     # 202
K = 40
N_REPS = 30                 # number of resampled training sets
BAG_B_DEPTH = 100           # ensemble size for the depth experiment (b)
SHALLOW_DEPTH, MID_DEPTH, DEEP_DEPTH = 2, 8, None   # depth levels for (b); MID only used if the selected depth coincides with another level
N_JOBS = -1                 # -1 = all cores, 1 = serial
B_LIST = [1, 5, 10, 25, 50, 100, 200]; DEPTHS_B = [1, 2, 3, 4]; ROUNDS = [10, 25, 50, 100]   # only for recomputing Q1/Q2 choices
print("SEED =", SEED)

In [ ]:
import os, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.datasets import fetch_olivetti_faces
from sklearn.tree import DecisionTreeClassifier

for d in ["data", "split", "logs", "figures"]:
    os.makedirs(d, exist_ok=True)
faces = fetch_olivetti_faces(data_home="data/sklearn_data", shuffle=False)
X = faces.data.astype(np.float64); y = faces.target.astype(int)
assert X.shape == (400, 4096)

for cand in ["split/split_indices.npz", "split_indices.npz"]:          # the SAME saved split as Q1 and Q2
    if os.path.exists(cand):
        s = np.load(cand); tr_idx, va_idx, te_idx = s["train"], s["val"], s["test"]; print("Loaded split from", cand); break
else:
    rng = np.random.default_rng(SEED); tr, va, te = [], [], []
    for k in range(K):
        p = rng.permutation(np.where(y == k)[0]); tr += list(p[:6]); va += list(p[6:8]); te += list(p[8:])
    tr_idx, va_idx, te_idx = np.array(tr), np.array(va), np.array(te)
    np.savez("split/split_indices.npz", train=tr_idx, val=va_idx, test=te_idx, seed=SEED); print("Created split")
assert len(set(tr_idx) | set(va_idx) | set(te_idx)) == 400

def mask_images(A):
    B = A.copy().reshape(-1, 64, 64); B[:, 24:40, 24:40] = 0.0
    return B.reshape(len(A), -1)

Xtr, ytr = X[tr_idx], y[tr_idx]; Xva, yva = X[va_idx], y[va_idx]; Xte, yte = X[te_idx], y[te_idx]
Xte_m = mask_images(Xte)
print(Xtr.shape, Xva.shape, Xte.shape, Xte_m.shape)

In [ ]:
# ---- my own bagging and SAMME (same code as Q1 / Q2) ----
class Bagging:
    def __init__(self, n_estimators, tree_params, seed):
        self.B, self.tree_params, self.seed = n_estimators, tree_params, seed
    def fit(self, X, y):
        rng = np.random.default_rng(self.seed); n = len(X)
        self.trees, self.bags = [], []
        for _ in range(self.B):
            idx = rng.integers(0, n, size=n)
            t = DecisionTreeClassifier(**self.tree_params, random_state=int(rng.integers(2**31 - 1)))
            self.trees.append(t.fit(X[idx], y[idx])); self.bags.append(idx)
        return self
    def tree_preds(self, X):
        return np.stack([t.predict(X) for t in self.trees])

def votes_from_preds(P, n_classes=K):
    V = np.zeros((P.shape[1], n_classes), int)
    for p in P: V[np.arange(len(p)), p] += 1
    return V

def disagreement_per_sample(P):
    B = P.shape[0]
    if B < 2: return np.full(P.shape[1], np.nan)
    V = votes_from_preds(P)
    return (B**2 - (V**2).sum(1)) / (B * (B - 1))

class SAMME:
    def __init__(self, n_estimators, max_depth, n_classes, seed, eps_floor=1e-10):
        self.T, self.depth, self.K, self.seed, self.eps_floor = n_estimators, max_depth, n_classes, seed, eps_floor
    def fit(self, X, y):
        n, K = len(y), self.K
        w = np.full(n, 1.0 / n); self.trees, self.alphas, self.events = [], [], []
        for t in range(self.T):
            tree = DecisionTreeClassifier(max_depth=self.depth, random_state=self.seed + t)
            tree.fit(X, y, sample_weight=w)
            wrong = tree.predict(X) != y
            eps = float(np.sum(w * wrong) / np.sum(w))
            if eps >= 1.0 - 1.0 / K:                    # not better than random K-class guessing: discard and stop
                self.events.append(f"round {t+1}: eps {eps:.4f} >= 1-1/K; discarded, stopped"); break
            eps_c = max(eps, self.eps_floor)
            alpha = float(np.log((1 - eps_c) / eps_c) + np.log(K - 1))
            w_new = w * np.exp(alpha * wrong); w_new /= w_new.sum()
            self.trees.append(tree); self.alphas.append(alpha)
            if eps == 0.0: self.events.append(f"round {t+1}: perfect learner, stopped"); break
            w = w_new
        self.alphas = np.array(self.alphas)
        return self
    def learner_preds(self, X):
        return np.stack([t.predict(X) for t in self.trees])

def samme_scores(P, alphas, n, K=K):
    S = np.zeros((P.shape[1], K))
    for p, a in zip(P[:n], alphas[:n]): S[np.arange(len(p)), p] += a
    return S

def samme_q(P, alphas, n, K=K):                          # normalised weighted vote vector
    return samme_scores(P, alphas, n, K) / np.sum(alphas[:n])

In [ ]:
# ---- frozen choices from Q1 and Q2 (never re-tuned here) ----
def load_or_compute_q1():
    try:
        return json.load(open("logs/frozen_tree_params.json")), int(json.load(open("logs/final_B.json"))["final_B"]), "Q1 logs"
    except FileNotFoundError:
        best, order = None, 0
        for d in [2, 4, 8, 16, None]:
            for m in [1, 2, 4]:
                c = DecisionTreeClassifier(max_depth=d, min_samples_leaf=m, random_state=SEED).fit(Xtr, ytr)
                key = (-c.score(Xva, yva), c.get_n_leaves(), order); order += 1
                if best is None or key < best[0]: best = (key, d, m)
        TP = dict(max_depth=best[1], min_samples_leaf=best[2])
        bg = Bagging(200, TP, SEED).fit(Xtr, ytr); Pv = bg.tree_preds(Xva)
        va = [(votes_from_preds(Pv[:B]).argmax(1) == yva).mean() for B in B_LIST]
        return TP, B_LIST[int(np.argmax(va))], "recomputed (Q1 rules)"

def load_or_compute_q2():
    try:
        j = json.load(open("logs/q2_frozen_boosting.json")); return int(j["depth"]), int(j["rounds"]), "Q2 logs"
    except FileNotFoundError:
        rows = []
        for d in DEPTHS_B:
            m = SAMME(max(ROUNDS), d, K, SEED).fit(Xtr, ytr); P = m.learner_preds(Xva)
            for R in ROUNDS:
                n = min(R, len(m.trees)); acc = (samme_scores(P, m.alphas, n).argmax(1) == yva).mean() if n else -1
                rows.append((-acc, d, R))
        _, d, R = min(rows); return d, R, "recomputed (Q2 rules)"

TREE_PARAMS, FINAL_B, s1 = load_or_compute_q1()
SEL_D, SEL_R, s2 = load_or_compute_q2()
SEL_DEPTH, SEL_MSL = TREE_PARAMS["max_depth"], TREE_PARAMS["min_samples_leaf"]
print(f"single tree {TREE_PARAMS} | bagging B={FINAL_B} ({s1}) | boosting depth={SEL_D}, rounds={SEL_R} ({s2})")

# depth levels for (b): shallow, selected, deepest. Only max_depth varies; min_samples_leaf stays at the selected value.
key_ = lambda d: 10**6 if d is None else d
LEVELS = []
for d in [SHALLOW_DEPTH, SEL_DEPTH, DEEP_DEPTH]:
    if d not in LEVELS: LEVELS.append(d)
if len(LEVELS) < 3:
    for d in [MID_DEPTH, 4, 16]:
        if d not in LEVELS: LEVELS.append(d); break
    print("NOTE: the selected depth coincides with the shallow/deep level, so an extra intermediate level was added.")
LEVELS.sort(key=key_)
ROLE = {d: ("selected (and deepest)" if d == SEL_DEPTH and d == LEVELS[-1] else "selected" if d == SEL_DEPTH else "shallow" if d == LEVELS[0] else "deep" if d == LEVELS[-1] else "intermediate") for d in LEVELS}
print("depth levels:", [(str(d), ROLE[d]) for d in LEVELS], "| min_samples_leaf =", SEL_MSL)

## Resampled training sets (fixed once and saved)
For every identity, 6 faces drawn with replacement from that identity's 6 original training faces. The 30 index sets are saved to `split/q3_resample_indices.npy`.

In [ ]:
RS_FILE = "split/q3_resample_indices.npy"
if os.path.exists(RS_FILE) and np.load(RS_FILE).shape == (N_REPS, len(ytr)):
    RESAMPLE = np.load(RS_FILE); print("Loaded resamples")
else:
    rng = np.random.default_rng(SEED); pos = [np.where(ytr == k)[0] for k in range(K)]
    RESAMPLE = np.stack([np.concatenate([rng.choice(pos[k], size=6, replace=True) for k in range(K)]) for _ in range(N_REPS)])
    np.save(RS_FILE, RESAMPLE); print("Created and saved resamples")
assert RESAMPLE.shape == (N_REPS, 240)
for r in range(N_REPS):
    assert (np.bincount(ytr[RESAMPLE[r]], minlength=K) == 6).all()          # 6 faces per identity, drawn only from that identity
print("distinct images per resampled set (mean):", np.mean([len(np.unique(a)) for a in RESAMPLE]), "| theory 240*(1-(5/6)^6) =", 240 * (1 - (5/6) ** 6))

## Refit all models on each resampled set (no tuning)
Class vectors: single tree = `predict_proba` (leaf class proportions); bagging = vote fractions; boosting = normalised weighted vote `S_k / sum(alpha)`. Bagging uses a different bootstrap seed in every repetition, so its own randomness is part of the measured variance.

In [ ]:
def structure_stats(t):
    tr = t.tree_; leaf = tr.children_left == -1; ns = tr.n_node_samples[leaf]
    return np.array([leaf.sum(), ns.mean(), (ns == 1).mean(), t.get_depth()], float)   # leaves, mean samples/leaf, frac singleton leaves, actual depth

def proba_full(t, A):
    P = np.zeros((len(A), K)); P[:, t.classes_] = t.predict_proba(A); return P

def run_rep(r):
    idx = RESAMPLE[r]; Xr, yr = Xtr[idx], ytr[idx]; out = {}; bag_seed = SEED + 10000 * (r + 1)
    for d in LEVELS:
        params = dict(max_depth=d, min_samples_leaf=SEL_MSL)
        t = DecisionTreeClassifier(**params, random_state=SEED).fit(Xr, yr)
        out[("tree", d)] = dict(clean=proba_full(t, Xte), masked=proba_full(t, Xte_m), struct=structure_stats(t))
        nB = max(BAG_B_DEPTH, FINAL_B) if d == SEL_DEPTH else BAG_B_DEPTH
        bg = Bagging(nB, params, bag_seed).fit(Xr, yr)
        Pc, Pm = bg.tree_preds(Xte), bg.tree_preds(Xte_m)
        st = np.mean([structure_stats(tt) for tt in bg.trees[:BAG_B_DEPTH]], axis=0)
        out[("bag", d)] = dict(clean=votes_from_preds(Pc[:BAG_B_DEPTH]) / BAG_B_DEPTH, masked=votes_from_preds(Pm[:BAG_B_DEPTH]) / BAG_B_DEPTH,
                               struct=st, disagree=float(disagreement_per_sample(Pc[:BAG_B_DEPTH]).mean()))
        if d == SEL_DEPTH:          # final bagged model = first FINAL_B trees (identical to Bagging(FINAL_B, ..., same seed))
            out[("bag_final",)] = dict(clean=votes_from_preds(Pc[:FINAL_B]) / FINAL_B, masked=votes_from_preds(Pm[:FINAL_B]) / FINAL_B,
                                       disagree=float(np.nanmean(disagreement_per_sample(Pc[:FINAL_B]))) if FINAL_B > 1 else float("nan"))
    bs = SAMME(SEL_R, SEL_D, K, SEED).fit(Xr, yr); n = len(bs.trees)
    if n: qc, qm = samme_q(bs.learner_preds(Xte), bs.alphas, n), samme_q(bs.learner_preds(Xte_m), bs.alphas, n)
    else: qc = qm = np.zeros((len(Xte), K))
    out[("boost",)] = dict(clean=qc, masked=qm, n_learners=n)
    return out

t0 = time.perf_counter()
outs = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(run_rep)(r) for r in range(N_REPS))
print(f"done: {N_REPS} refits in {time.perf_counter() - t0:.0f} s")

def G(cfg, which): return np.stack([o[cfg][which] for o in outs])           # (R, N, K)
E = np.eye(K)[yte]

def bias_var(P):
    pbar = P.mean(0)
    bias2 = ((pbar - E) ** 2).sum(1).mean(); var = ((P - pbar) ** 2).sum(2).mean()
    mse = ((P - E) ** 2).sum(2).mean()
    assert np.isclose(mse, bias2 + var)                                      # decomposition identity: mean squared error = bias^2 + variance
    return float(bias2), float(var)

def acc_reps(P): return (P.argmax(2) == yte).mean(1)                         # (R,)
print("boosting learners kept per refit: min", min(o[("boost",)]["n_learners"] for o in outs), "max", max(o[("boost",)]["n_learners"] for o in outs))

## Q3(a) Selected single tree, final bagged model, selected boosting model

In [ ]:
SD = lambda a: float(np.std(a, ddof=1))
tab_a = []
for name, cfg in [("single tree", ("tree", SEL_DEPTH)), (f"bagging (B={FINAL_B})", ("bag_final",)), (f"boosting (depth {SEL_D}, {SEL_R} rounds)", ("boost",))]:
    P = G(cfg, "clean"); b2, v = bias_var(P); a = acc_reps(P)
    tab_a.append(dict(model=name, bias2=b2, variance=v, bias2_plus_var=b2 + v, acc_mean=a.mean(), acc_std=SD(a)))
# supplementary: single tree with hard one-hot predictions instead of leaf probabilities
Ph = np.eye(K)[G(("tree", SEL_DEPTH), "clean").argmax(2)]; b2, v = bias_var(Ph)
tab_a.append(dict(model="single tree (hard one-hot vector, supplementary)", bias2=b2, variance=v, bias2_plus_var=b2 + v, acc_mean=acc_reps(Ph).mean(), acc_std=SD(acc_reps(Ph))))
tab_a = pd.DataFrame(tab_a); tab_a.to_csv("logs/q3a_selected_models.csv", index=False); display(tab_a.round(4))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2)); x = np.arange(3); w_ = .38; t3 = tab_a.iloc[:3]
ax[0].bar(x - w_/2, t3.bias2, w_, label="bias^2"); ax[0].bar(x + w_/2, t3.variance, w_, label="variance")
ax[0].set_xticks(x); ax[0].set_xticklabels(["tree", "bagging", "boosting"]); ax[0].legend(); ax[0].set_title("measured bias^2 and variance", fontsize=9)
ax[1].bar(x, t3.acc_mean, yerr=t3.acc_std, capsize=4); ax[1].set_xticks(x); ax[1].set_xticklabels(["tree", "bagging", "boosting"])
ax[1].set_title(f"clean test accuracy, mean +- sd over {N_REPS} refits", fontsize=9)
plt.tight_layout(); plt.savefig("figures/q3a_bias_variance.pdf"); plt.show()

## Q3(b) Depth sweep: single trees vs 100-tree bagged ensembles

In [ ]:
rows_b = []
for d in LEVELS:
    for model, cfg in [("single tree", ("tree", d)), (f"bagging B={BAG_B_DEPTH}", ("bag", d))]:
        P = G(cfg, "clean"); b2, v = bias_var(P); a = acc_reps(P); st = np.mean([o[cfg]["struct"] for o in outs], axis=0)
        rows_b.append(dict(depth=str(d), role=ROLE[d], model=model, bias2=b2, variance=v, bias2_plus_var=b2 + v, acc_mean=a.mean(), acc_std=SD(a),
                           n_leaves=st[0], mean_samples_per_leaf=st[1], frac_singleton_leaves=st[2], actual_depth=st[3]))
tab_b = pd.DataFrame(rows_b); tab_b.to_csv("logs/q3b_depth_sweep.csv", index=False); display(tab_b.round(4))

fig, ax = plt.subplots(1, 3, figsize=(12, 3.2)); xl = [str(d) for d in LEVELS]
for col, a_, ttl in [("bias2", ax[0], "bias^2"), ("variance", ax[1], "variance"), ("acc_mean", ax[2], "clean test accuracy")]:
    for model, mk in [("single tree", "o-"), (f"bagging B={BAG_B_DEPTH}", "s--")]:
        g = tab_b[tab_b.model == model]; a_.plot(g.depth, g[col], mk, label=model)
    a_.set_title(ttl, fontsize=9); a_.set_xlabel("max_depth"); a_.grid(alpha=.3)
ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig("figures/q3b_depth_sweep.pdf"); plt.show()

## Q3(c) Is the deepest tree the depth where bagging helps most? (measured evidence)

In [ ]:
rows_c = []
for d in LEVELS:
    Pt, Pb = G(("tree", d), "clean"), G(("bag", d), "clean")
    bt, vt = bias_var(Pt); bb, vb = bias_var(Pb); at, ab = acc_reps(Pt), acc_reps(Pb); dacc = ab - at       # paired per-refit difference
    st = np.mean([o[("tree", d)]["struct"] for o in outs], axis=0); sb = np.mean([o[("bag", d)]["struct"] for o in outs], axis=0)
    dis = np.mean([o[("bag", d)]["disagree"] for o in outs])
    rows_c.append(dict(depth=str(d), role=ROLE[d], tree_var=vt, bag_var=vb, var_removed=vt - vb, var_removed_pct=100 * (vt - vb) / vt,
                       tree_bias2=bt, bag_bias2=bb, bias2_change=bb - bt, tree_acc=at.mean(), bag_acc=ab.mean(),
                       acc_gain=dacc.mean(), acc_gain_sd=SD(dacc),
                       tree_leaves=st[0], tree_mean_samples_per_leaf=st[1], tree_frac_singleton_leaves=st[2], tree_actual_depth=st[3],
                       bag_trees_mean_samples_per_leaf=sb[1], bag_trees_frac_singleton_leaves=sb[2], bag_pairwise_disagreement=dis))
tab_c = pd.DataFrame(rows_c); tab_c.to_csv("logs/q3c_bagging_gain_by_depth.csv", index=False); display(tab_c.round(4).T)

deepest = tab_c.iloc[-1].depth
print("level with the largest accuracy gain from bagging :", tab_c.loc[tab_c.acc_gain.idxmax(), "depth"], f"(gain {tab_c.acc_gain.max():.4f})")
print("level with the largest variance removed           :", tab_c.loc[tab_c.var_removed.idxmax(), "depth"], f"({tab_c.var_removed.max():.4f})")
print("deepest level                                     :", deepest, "-> deepest level has the largest accuracy gain:", tab_c.acc_gain.idxmax() == len(tab_c) - 1)
print("gains are mean +- sd of paired per-refit differences; with", N_REPS, "refits and 80 test faces, differences within about one sd are not conclusive.")

## Q3(d) Final table and sensitivity to the missing region

In [ ]:
rows_d = []
for name, cfg in [("single tree", ("tree", SEL_DEPTH)), (f"bagging (B={FINAL_B})", ("bag_final",)), (f"boosting (depth {SEL_D}, {SEL_R} rounds)", ("boost",))]:
    Pc, Pm = G(cfg, "clean"), G(cfg, "masked"); b2, v = bias_var(Pc); b2m, vm = bias_var(Pm); ac, am = acc_reps(Pc), acc_reps(Pm)
    rows_d.append(dict(model=name, clean_acc=ac.mean(), clean_acc_sd=SD(ac), masked_acc=am.mean(), masked_acc_sd=SD(am), acc_drop=ac.mean() - am.mean(),
                       bias2=b2, variance=v, masked_bias2=b2m, masked_variance=vm,
                       class_vector_shift=float(((Pc - Pm) ** 2).sum(2).mean()),            # mean ||p(clean) - p(masked)||^2
                       prediction_flip_rate=float((Pc.argmax(2) != Pm.argmax(2)).mean()),
                       mean_winning_vote_clean=float(Pc.max(2).mean()), mean_winning_vote_masked=float(Pm.max(2).mean())))
tab_d = pd.DataFrame(rows_d); tab_d.to_csv("logs/q3d_final_table.csv", index=False)
display(tab_d[["model", "clean_acc", "masked_acc", "bias2", "variance"]].round(4))      # the requested table
display(tab_d.drop(columns=["clean_acc", "masked_acc", "bias2", "variance"]).round(4))   # supplementary columns for the explanation

per_rep = pd.DataFrame({"rep": range(N_REPS)})
for nm, cfg in [("tree", ("tree", SEL_DEPTH)), ("bagging", ("bag_final",)), ("boosting", ("boost",))]:
    per_rep[f"{nm}_clean_acc"] = acc_reps(G(cfg, "clean")); per_rep[f"{nm}_masked_acc"] = acc_reps(G(cfg, "masked"))
per_rep.to_csv("logs/q3_per_refit_accuracy.csv", index=False)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.3)); nm3 = ["tree", "bagging", "boosting"]
ax[0].boxplot([per_rep[f"{n}_clean_acc"] for n in nm3] + [per_rep[f"{n}_masked_acc"] for n in nm3])
ax[0].set_xticklabels([f"{n}\nclean" for n in nm3] + [f"{n}\nmasked" for n in nm3])
ax[0].set_title("test accuracy over refits", fontsize=9); ax[0].tick_params(axis="x", labelsize=6)
xx = np.arange(3); ax[1].bar(xx - .2, tab_d.bias2, .4, label="bias^2 (clean)"); ax[1].bar(xx + .2, tab_d.variance, .4, label="variance (clean)")
ax[1].set_xticks(xx); ax[1].set_xticklabels(nm3); ax[1].legend(fontsize=7); ax[1].set_title("bias^2 and variance", fontsize=9)
ax[2].bar(xx - .2, tab_d.acc_drop, .4, label="accuracy drop"); ax[2].bar(xx + .2, tab_d.prediction_flip_rate, .4, label="prediction flip rate")
ax[2].set_xticks(xx); ax[2].set_xticklabels(nm3); ax[2].legend(fontsize=7); ax[2].set_title("sensitivity to the masked block", fontsize=9)
plt.tight_layout(); plt.savefig("figures/q3d_stability_sensitivity.pdf"); plt.show()

In [ ]:
final = dict(roll_no=ROLL_NO, seed=SEED, n_reps=N_REPS, frozen_tree=TREE_PARAMS, final_B=FINAL_B, boosting=dict(depth=SEL_D, rounds=SEL_R),
             depth_levels=[str(d) for d in LEVELS], bag_size_depth_experiment=BAG_B_DEPTH,
             table_a=tab_a.to_dict(orient="records"), table_b=tab_b.to_dict(orient="records"),
             table_c=tab_c.to_dict(orient="records"), table_d=tab_d.to_dict(orient="records"))
json.dump(final, open("logs/q3_final_log.json", "w"), indent=2, default=str); print("saved logs/q3_final_log.json")